In [3]:
from pathlib import Path
import re
import shutil
import pandas as pd


# ====== Konfigurasi ======
SPLIT_DIR = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\dataset\split")
CLIENT_DIR = SPLIT_DIR / "clientsMalacious"
NEW_CLIENT_DIR = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2") / "clientsMalacious_Selected"

TOP21_STATES = [
    53, 36, 24, 12, 39, 27, 55,
    31, 18, 23, 26, 20, 48, 45,
    19, 17, 4, 13, 49, 51, 54,
]

SPLITS = ("train", "val")
REQUIRED_FILES = tuple(f"{s}.csv" for s in SPLITS)
COPY_FILES = True   # salin CSV ke folder baru
OVERWRITE = False   # True = timpa file yang sudah ada di folder baru

if not CLIENT_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {CLIENT_DIR}")

# ====== 1. Deteksi semua state yang ada di disk ======
all_states = sorted(
    int(m.group(1))
    for p in CLIENT_DIR.iterdir()
    if p.is_dir() and (m := re.fullmatch(r"state_(\d+)", p.name))
)
NEW_STATES = [s for s in all_states if s not in TOP21_STATES]

print(f"State di disk : {len(all_states)}")
print(f"TOP21         : {len(TOP21_STATES)}")
print(f"Client baru   : {len(NEW_STATES)} -> {NEW_STATES}")

# ====== 2. Validasi 21 client terpilih ======
selected_clients = {s: CLIENT_DIR / f"state_{s:02d}" for s in TOP21_STATES}

missing = [
    str(path / f)
    for path in selected_clients.values()
    for f in REQUIRED_FILES
    if not (path / f).is_file()
]
if missing:
    raise FileNotFoundError("File berikut tidak ditemukan:\n" + "\n".join(missing))

# ====== 3. Muat data 21 client (train & val) ======
clients_data = {
    s: {split: pd.read_csv(path / f"{split}.csv") for split in SPLITS}
    for s, path in selected_clients.items()
}

# ====== 4. Ringkasan ======
print(f"\nBerhasil memuat {len(clients_data)} client.\n")
print(f"{'State':<8}{'Train':>10}{'Val':>10}{'Kolom':>8}")
for s, d in clients_data.items():
    print(
        f"{s:02d}".ljust(8)
        + f"{len(d['train']):>10}{len(d['val']):>10}"
        + f"{d['train'].shape[1]:>8}"
    )

# ====== 5. Cek konsistensi kolom ======
ref_cols = list(clients_data[TOP21_STATES[0]]["train"].columns)
for s, d in clients_data.items():
    for split, df in d.items():
        if list(df.columns) != ref_cols:
            print(f"⚠️ Kolom state_{s:02d}/{split} berbeda dari client pertama")

# ====== 6. Buat folder client baru + salin CSV ======
NEW_CLIENT_DIR.mkdir(parents=True, exist_ok=True)

copied = skipped = absent = 0
for s in NEW_STATES:
    src_dir = CLIENT_DIR / f"state_{s:02d}"
    dst_dir = NEW_CLIENT_DIR / f"state_{s:02d}"
    dst_dir.mkdir(exist_ok=True)

    if not COPY_FILES:
        continue
    for f in REQUIRED_FILES:
        src, dst = src_dir / f, dst_dir / f
        if not src.is_file():
            absent += 1
            print(f"⚠️ Tidak ada di sumber: state_{s:02d}/{f}")
        elif dst.exists() and not OVERWRITE:
            skipped += 1
        else:
            shutil.copy2(src, dst)
            copied += 1

# ====== 7. Verifikasi hasil ======
n_csv = sum(
    1
    for p in NEW_CLIENT_DIR.glob("state_*")
    for f in REQUIRED_FILES
    if (p / f).is_file()
)
print(f"\nFolder client baru : {NEW_CLIENT_DIR}")
print(f"Folder state       : {len(NEW_STATES)}")
print(f"CSV disalin        : {copied} | dilewati (sudah ada): {skipped} | tidak ada di sumber: {absent}")
print(f"Total train/val CSV di folder baru: {n_csv}")

State di disk : 44
TOP21         : 21
Client baru   : 23 -> [1, 2, 5, 8, 10, 11, 15, 16, 22, 25, 29, 30, 33, 34, 35, 37, 38, 40, 41, 44, 46, 50, 56]

Berhasil memuat 21 client.

State        Train       Val   Kolom
53            3531      1850      72
36            2212      3166      72
24            2845      1310      72
12            2330      1257      72
39            2757      1057      72
27            2163      1047      72
55            1605      1509      72
31            1790      1310      72
18            1906      1350      72
23            1763      1003      72
26            1525      1079      72
20            1684       914      72
48            1987       998      72
45            1812      1044      72
19            1589       870      72
17             727      1224      72
04            1781       804      72
13            1605       695      72
49            1176       759      72
51            1567       673      72
54            1101       913      72

Folder 

In [5]:
from LabelReverse import reverse_client_labels, compare_outputs

CLIENTS_DIR = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2\clientsMalacious_Selected"
NEW_DIR     = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2\clientsMalacious_Reverse"

reverse_client_labels(CLIENTS_DIR, label_col="_michd", output_dir=NEW_DIR)
compare_outputs(CLIENTS_DIR, NEW_DIR, label_col="_michd")

state_01
  train.csv: {0: 743, 1: 236} -> {0: 236, 1: 743}
  val.csv: {0: 438, 1: 124} -> {0: 124, 1: 438}
state_02
  train.csv: {0: 518, 1: 126} -> {0: 126, 1: 518}
  val.csv: {0: 291, 1: 67} -> {0: 67, 1: 291}
state_05
  train.csv: {0: 797, 1: 279} -> {0: 279, 1: 797}
  val.csv: {0: 441, 1: 154} -> {0: 154, 1: 441}
state_08
  train.csv: {0: 776, 1: 151} -> {0: 151, 1: 776}
  val.csv: {0: 513, 1: 133} -> {0: 133, 1: 513}
state_10
  train.csv: {0: 542, 1: 144} -> {0: 144, 1: 542}
  val.csv: {0: 318, 1: 96} -> {0: 96, 1: 318}
state_11
  train.csv: {0: 296, 1: 49} -> {0: 49, 1: 296}
  val.csv: {0: 170, 1: 29} -> {0: 29, 1: 170}
state_15
  train.csv: {0: 1126, 1: 235} -> {0: 235, 1: 1126}
  val.csv: {0: 484, 1: 107} -> {0: 107, 1: 484}
state_16
  train.csv: {0: 635, 1: 156} -> {0: 156, 1: 635}
  val.csv: {0: 220, 1: 63} -> {0: 63, 1: 220}
state_22
  train.csv: {0: 788, 1: 221} -> {0: 221, 1: 788}
  val.csv: {0: 323, 1: 92} -> {0: 92, 1: 323}
state_25
  train.csv: {0: 819, 1: 227} -> {0: 2

,state,split,asli_0,asli_1,baru_0,baru_1,semua_baris_terbalik
0,01,train,743,236,236,743,True
1,01,val,438,124,124,438,True
2,02,train,518,126,126,518,True
3,02,val,291,67,67,291,True
4,05,train,797,279,279,797,True
5,05,val,441,154,154,441,True
6,08,train,776,151,151,776,True
7,08,val,513,133,133,513,True
8,10,train,542,144,144,542,True
9,10,val,318,96,96,318,True
